##Task2:

In [1]:
import heapq


# Warehouse map from the laboratory sheet
warehouse = [
    "#################",
    "#S....#.........#",
    "#.###.#.#######.#",
    "#...#.#.......#.#",
    "###.#.#######.#.#",
    "#...#.........#.#",
    "#.###########.#.#",
    "#.............#G#",
    "#################"
]


# Available actions:
# Up, Down, Left, Right
directions = [
    (-1, 0),  # Up
    (1, 0),   # Down
    (0, -1),  # Left
    (0, 1)    # Right
]


def find_position(grid, symbol):
    """Find the position of a symbol in the warehouse."""
    for r, row in enumerate(grid):
        for c, cell in enumerate(row):
            if cell == symbol:
                return (r, c)

    return None


def heuristic(position, goal):
    """
    Manhattan distance heuristic.

    h(n) = |x - xG| + |y - yG|
    """
    return abs(position[0] - goal[0]) + abs(position[1] - goal[1])


def reconstruct_path(came_from, current):
    """Reconstruct the path from the goal back to the start."""

    path = [current]

    while current in came_from:
        current = came_from[current]
        path.append(current)

    path.reverse()

    return path


def astar(grid):
    """
    Perform A* search from S to G.

    Returns:
        path: list of positions if a solution exists
        states_expanded: number of expanded states
    """

    start = find_position(grid, "S")
    goal = find_position(grid, "G")

    if start is None or goal is None:
        return None, 0

    # Frontier implemented as a priority queue.
    # Each entry contains:
    # (f(n), g(n), state)
    frontier = []

    initial_h = heuristic(start, goal)

    heapq.heappush(
        frontier,
        (initial_h, 0, start)
    )

    # Store the predecessor of each state
    came_from = {
        start: None
    }

    # g(n): cost from start to each state
    g_cost = {
        start: 0
    }

    states_expanded = 0

    while frontier:

        # Select the state with the smallest f(n)
        f, current_g, current = heapq.heappop(frontier)

        # Ignore outdated entries
        if current_g != g_cost[current]:
            continue

        states_expanded += 1

        # Goal test
        if current == goal:
            path = reconstruct_path(
                came_from,
                current
            )

            return path, states_expanded

        r, c = current

        # Try all four possible actions
        for dr, dc in directions:

            new_r = r + dr
            new_c = c + dc

            # Check whether the position is inside the grid
            if not (0 <= new_r < len(grid)):
                continue

            if not (0 <= new_c < len(grid[0])):
                continue

            # Check whether the cell is an obstacle
            if grid[new_r][new_c] == "#":
                continue

            neighbor = (new_r, new_c)

            # Every movement has cost 1
            new_g = current_g + 1

            # Update if this is the best path found so far
            if new_g < g_cost.get(
                neighbor,
                float("inf")
            ):

                g_cost[neighbor] = new_g

                came_from[neighbor] = current

                h = heuristic(
                    neighbor,
                    goal
                )

                f = new_g + h

                heapq.heappush(
                    frontier,
                    (f, new_g, neighbor)
                )

    # No path exists
    return None, states_expanded


def print_path(grid, path):
    """Print the warehouse with the discovered path."""

    if path is None:
        print("No path exists from S to G.")
        return

    path_set = set(path)

    for r, row in enumerate(grid):

        line = ""

        for c, cell in enumerate(row):

            if (r, c) in path_set and cell not in ("S", "G"):
                line += "*"
            else:
                line += cell

        print(line)


# Run A*
path, states_expanded = astar(warehouse)


# Report results
if path is not None:

    print("Solution found!")
    print("Path length:", len(path) - 1)
    print("States expanded:", states_expanded)

    print("\nPath:")
    print(path)

    print("\nWarehouse with path:")
    print_path(warehouse, path)

else:

    print("No solution found.")
    print("States expanded:", states_expanded)

Solution found!
Path length: 41
States expanded: 64

Path:
[None, (1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (5, 6), (5, 7), (5, 8), (5, 9), (5, 10), (5, 11), (5, 12), (5, 13), (4, 13), (3, 13), (3, 12), (3, 11), (3, 10), (3, 9), (3, 8), (3, 7), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (2, 15), (3, 15), (4, 15), (5, 15), (6, 15), (7, 15)]

Warehouse with path:
#################
#S****#*********#
#.###*#*#######*#
#...#*#*******#*#
###.#*#######*#*#
#...#*********#*#
#.###########.#*#
#.............#G#
#################


**Observation:** The generated program found a path, but inspection of
the reconstructed path revealed that `None` was added when the
predecessor of the start state was followed. After the path was
reversed, `None` appeared at the beginning of the returned path. This
caused the reported path length to be one greater than the actual number
of movements.

### Bug Fix

The problem was in the path reconstruction logic. The start state has
`None` as its predecessor:

`came_from[start] = None`

However, the original loop continued while the current state was present
in `came_from`, causing the `None` predecessor to be added to the path.

The reconstruction logic was therefore changed to stop when the start
state is reached.

In [3]:
def reconstruct_path(came_from, current):
    """Reconstruct the path from the goal back to the start."""

    path = [current]

    while came_from[current] is not None:
        current = came_from[current]
        path.append(current)

    path.reverse()

    return path

path, states_expanded = astar(warehouse)

print("Solution found!")
print("Path length:", len(path) - 1)
print("States expanded:", states_expanded)

print("\nPath:")
print(path)

print("\nWarehouse with path:")
print_path(warehouse, path)



Solution found!
Path length: 40
States expanded: 64

Path:
[(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (5, 6), (5, 7), (5, 8), (5, 9), (5, 10), (5, 11), (5, 12), (5, 13), (4, 13), (3, 13), (3, 12), (3, 11), (3, 10), (3, 9), (3, 8), (3, 7), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (2, 15), (3, 15), (4, 15), (5, 15), (6, 15), (7, 15)]

Warehouse with path:
#################
#S****#*********#
#.###*#*#######*#
#...#*#*******#*#
###.#*#######*#*#
#...#*********#*#
#.###########.#*#
#.............#G#
#################


**Test 1 Result:** The corrected program successfully found a
collision-free path from `S` to `G`. The final path contains 40 moves,
and 64 states were expanded during the search. The reconstructed path
now starts at `(1, 1)` and ends at `(7, 15)` without containing the
invalid `None` value.

## Test 2 — Trivial Case

A very small warehouse is used in which the goal is immediately
adjacent to the starting position.

The expected solution is a single movement from `S` to `G`.

In [4]:
# Test 2: Trivial case

trivial_warehouse = [
    "#####",
    "#SG##",
    "#####"
]

path, states_expanded = astar(trivial_warehouse)

if path is not None:
    print("Solution found!")
    print("Path length:", len(path) - 1)
    print("States expanded:", states_expanded)

    print("\nPath:")
    print(path)

    print("\nWarehouse with path:")
    print_path(trivial_warehouse, path)

else:
    print("No solution found.")
    print("States expanded:", states_expanded)

Solution found!
Path length: 1
States expanded: 2

Path:
[(1, 1), (1, 2)]

Warehouse with path:
#####
#SG##
#####


**Test 2 Result:** The program correctly found the one-step solution.
The path length is 1 and two states were expanded: the starting state
and the goal state.

## Test 3 — No Solution

A warehouse is constructed in which the goal is completely
inaccessible from the starting position.

The purpose of this test is to verify that the program reports failure
and terminates rather than entering an infinite loop.

In [5]:
# Test 3: No-solution case

no_solution_warehouse = [
    "#######",
    "#S....#",
    "###.###",
    "#...#G#",
    "#######"
]

path, states_expanded = astar(no_solution_warehouse)

if path is not None:
    print("Solution found!")
    print("Path length:", len(path) - 1)
    print("States expanded:", states_expanded)

    print("\nPath:")
    print(path)

else:
    print("No solution found.")
    print("States expanded:", states_expanded)

No solution found.
States expanded: 9


**Test 3 Result:** The program correctly determined that no path exists.
It terminated after expanding 9 states and did not enter an infinite
loop.

## Test 4 — Alternative Paths

A warehouse containing multiple possible paths from `S` to `G` is used.
The direct path along the upper row has length 4, while routes through
the lower part of the warehouse are longer.

The purpose of this test is to determine whether A* returns the
shortest path.

In [6]:
# Test 4: Alternative paths

alternative_warehouse = [
    "#######",
    "#S...G#",
    "#.#.#.#",
    "#.....#",
    "#######"
]

path, states_expanded = astar(alternative_warehouse)

if path is not None:
    print("Solution found!")
    print("Path length:", len(path) - 1)
    print("States expanded:", states_expanded)

    print("\nPath:")
    print(path)

    print("\nWarehouse with path:")
    print_path(alternative_warehouse, path)

else:
    print("No solution found.")
    print("States expanded:", states_expanded)

Solution found!
Path length: 4
States expanded: 5

Path:
[(1, 1), (1, 2), (1, 3), (1, 4), (1, 5)]

Warehouse with path:
#######
#S***G#
#.#.#.#
#.....#
#######


**Test 4 Result:** The program returned a path of length 4, which is the
shortest possible path from `S` to `G`. This confirms that the A*
implementation selected the shortest path for this test case.

##Task5:

In [8]:
from collections import deque

def bfs(grid):
    start = find_position(grid, "S")
    goal = find_position(grid, "G")

    frontier = deque([start])
    came_from = {start: None}

    states_expanded = 0

    while frontier:
        current = frontier.popleft()
        states_expanded += 1

        if current == goal:
            path = reconstruct_path(came_from, current)
            return path, states_expanded

        r, c = current

        for dr, dc in directions:
            new_r = r + dr
            new_c = c + dc

            if not (0 <= new_r < len(grid)):
                continue

            if not (0 <= new_c < len(grid[0])):
                continue

            if grid[new_r][new_c] == "#":
                continue

            neighbor = (new_r, new_c)

            if neighbor not in came_from:
                came_from[neighbor] = current
                frontier.append(neighbor)

    return None, states_expanded

path_bfs, states_bfs = bfs(warehouse)

if path_bfs is not None:
    print("Solution found!")
    print("Path length:", len(path_bfs) - 1)
    print("States expanded:", states_bfs)

    print("\nPath:")
    print(path_bfs)

    print("\nWarehouse with path:")
    print_path(warehouse, path_bfs)
else:
    print("No solution found.")
    print("States expanded:", states_bfs)

Solution found!
Path length: 40
States expanded: 64

Path:
[(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (5, 6), (5, 7), (5, 8), (5, 9), (5, 10), (5, 11), (5, 12), (5, 13), (4, 13), (3, 13), (3, 12), (3, 11), (3, 10), (3, 9), (3, 8), (3, 7), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (2, 15), (3, 15), (4, 15), (5, 15), (6, 15), (7, 15)]

Warehouse with path:
#################
#S****#*********#
#.###*#*#######*#
#...#*#*******#*#
###.#*#######*#*#
#...#*********#*#
#.###########.#*#
#.............#G#
#################


##Task6:


In [9]:
import heapq
import math

def manhattan_heuristic(position, goal):
    return abs(position[0] - goal[0]) + abs(position[1] - goal[1])


def zero_heuristic(position, goal):
    return 0


def euclidean_heuristic(position, goal):
    return math.sqrt(
        (position[0] - goal[0]) ** 2 +
        (position[1] - goal[1]) ** 2
    )


def double_manhattan_heuristic(position, goal):
    return 2 * manhattan_heuristic(position, goal)

def astar_with_heuristic(grid, heuristic_function):
    start = find_position(grid, "S")
    goal = find_position(grid, "G")

    if start is None or goal is None:
        return None, 0

    frontier = []

    initial_h = heuristic_function(start, goal)

    heapq.heappush(
        frontier,
        (initial_h, 0, start)
    )

    came_from = {start: None}
    g_cost = {start: 0}

    states_expanded = 0

    while frontier:
        f, current_g, current = heapq.heappop(frontier)

        # Ignore outdated frontier entries
        if current_g != g_cost[current]:
            continue

        states_expanded += 1

        # Goal test
        if current == goal:
            path = reconstruct_path(came_from, current)
            return path, states_expanded

        r, c = current

        # Generate valid successors
        for dr, dc in directions:
            new_r = r + dr
            new_c = c + dc

            # Check boundaries
            if not (0 <= new_r < len(grid)):
                continue

            if not (0 <= new_c < len(grid[0])):
                continue

            # Check obstacle
            if grid[new_r][new_c] == "#":
                continue

            neighbor = (new_r, new_c)

            # Every movement has cost 1
            new_g = current_g + 1

            # Update only if a better path is found
            if new_g < g_cost.get(neighbor, float("inf")):

                g_cost[neighbor] = new_g
                came_from[neighbor] = current

                h = heuristic_function(neighbor, goal)
                f = new_g + h

                heapq.heappush(
                    frontier,
                    (f, new_g, neighbor)
                )

    return None, states_expanded

heuristics = [
    ("Manhattan", manhattan_heuristic),
    ("h(n) = 0", zero_heuristic),
    ("Euclidean", euclidean_heuristic),
    ("2 * Manhattan", double_manhattan_heuristic)
]

results = []

for name, heuristic_function in heuristics:

    path, states_expanded = astar_with_heuristic(
        warehouse,
        heuristic_function
    )

    solution_found = path is not None
    path_length = len(path) - 1 if path is not None else None

    results.append(
        (name, solution_found, path_length, states_expanded)
    )

    print("=" * 50)
    print("Heuristic:", name)
    print("Solution found:", solution_found)

    if path is not None:
        print("Path length:", path_length)

    print("States expanded:", states_expanded)

import pandas as pd

results_df = pd.DataFrame(
    results,
    columns=[
        "Heuristic",
        "Solution Found",
        "Path Length",
        "States Expanded"
    ]
)

results_df

for name, heuristic_function in heuristics:

    path, states_expanded = astar_with_heuristic(
        warehouse,
        heuristic_function
    )

    print("=" * 50)
    print(name)

    if path is not None:
        print("Path:", path)
        print("Path length:", len(path) - 1)
        print("States expanded:", states_expanded)
    else:
        print("No solution found.")
        print("States expanded:", states_expanded)

Heuristic: Manhattan
Solution found: True
Path length: 40
States expanded: 64
Heuristic: h(n) = 0
Solution found: True
Path length: 40
States expanded: 64
Heuristic: Euclidean
Solution found: True
Path length: 40
States expanded: 64
Heuristic: 2 * Manhattan
Solution found: True
Path length: 40
States expanded: 67
Manhattan
Path: [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (5, 6), (5, 7), (5, 8), (5, 9), (5, 10), (5, 11), (5, 12), (5, 13), (4, 13), (3, 13), (3, 12), (3, 11), (3, 10), (3, 9), (3, 8), (3, 7), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (2, 15), (3, 15), (4, 15), (5, 15), (6, 15), (7, 15)]
Path length: 40
States expanded: 64
h(n) = 0
Path: [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (5, 6), (5, 7), (5, 8), (5, 9), (5, 10), (5, 11), (5, 12), (5, 13), (4, 13), (3, 13), (3, 12), (3, 11), (3, 10), (3, 9), (3, 8), (3, 7), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12),